In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd

from sklearn.model_selection import train_test_split

In [ ]:
data_train = pd.read_csv('drive/MyDrive/ml_homework/hw4/data/train.csv')
data_train.head()

,text,label
0,— Кровь! какую кровь? — встревожилась,1
1,– Под нижнюю подушку.,0
2,— Благодарю-с...,1
3,— Когда же это-с?,1
4,"Старуха помолчала, как бы в раздумье,",1


In [ ]:
X_train, X_test = train_test_split(data_train, test_size=0.3)

In [ ]:
!pip install datasets
!pip install transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 11.2 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.10.0
    Uninstalling fsspec-2024.10.0:
      Successfully uninstalled fsspec-2024.10.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2024.9.0 which is incompatible.


In [ ]:
from datasets import Dataset

In [ ]:
X_train_dataset = Dataset.from_pandas(X_train, preserve_index=False)
X_test_dataset = Dataset.from_pandas(X_test, preserve_index=False)

In [ ]:
from transformers import AutoTokenizer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("distilbert/distilroberta-base")

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

tokenized_X_train = X_train_dataset.map(tokenize_function, batched=True)
tokenized_X_test = X_test_dataset.map(tokenize_function, batched=True)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Map:   0%|          | 0/4136 [00:00<?, ? examples/s]

Map:   0%|          | 0/1773 [00:00<?, ? examples/s]

In [ ]:
X_train = tokenized_X_train.remove_columns('text')
X_test = tokenized_X_test.remove_columns('text')

In [ ]:
from transformers import AutoModelForSequenceClassification

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained("distilbert/distilroberta-base", num_labels=2)

model.safetensors:   0%|          | 0.00/331M [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at distilbert/distilroberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
from transformers import TrainingArguments

In [ ]:
training_args = TrainingArguments(output_dir="\test_trainer")

In [ ]:
from transformers import TrainingArguments, Trainer

In [ ]:
training_args = TrainingArguments(output_dir="test_trainer", eval_strategy="epoch", report_to="none")

In [ ]:
training_args = TrainingArguments(
    output_dir="/test_trainer",
    eval_strategy="epoch",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    fp16=True,
    dataloader_num_workers=6,
    gradient_accumulation_steps=2
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=X_train,
    eval_dataset=X_test
)

In [ ]:
trainer.train()

/usr/local/lib/python3.10/dist-packages/torch/utils/data/dataloader.py:617: UserWarning: This DataLoader will create 6 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter, or press ctrl+c to quit:

 ··········


wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


/usr/local/lib/python3.10/dist-packages/torch/utils/data/dataloader.py:617: UserWarning: This DataLoader will create 6 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


Epoch,Training Loss,Validation Loss
0,No log,0.224827
2,No log,0.196554


/usr/local/lib/python3.10/dist-packages/torch/utils/data/dataloader.py:617: UserWarning: This DataLoader will create 6 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


TrainOutput(global_step=387, training_loss=0.23425637967210716, metrics={'train_runtime': 195.8292, 'train_samples_per_second': 63.361, 'train_steps_per_second': 1.976, 'total_flos': 1638356786577408.0, 'train_loss': 0.23425637967210716, 'epoch': 2.988416988416988})

In [ ]:
predictions = trainer.predict(X_test)

In [ ]:
predicted_labels = predictions.predictions.argmax(-1)

In [ ]:
predicted_labels

array([0, 1, 0, ..., 1, 1, 1])

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

In [ ]:
X_test_labels = X_test['label']

In [ ]:
accuracy = accuracy_score(X_test_labels, predicted_labels)
f1 = f1_score(X_test_labels, predicted_labels)
conf_matrix = confusion_matrix(X_test_labels, predicted_labels)

In [ ]:
accuracy

0.901297236322617

In [ ]:
f1

0.9031543995572773

In [ ]:
conf_matrix

array([[782,  83],
       [ 92, 816]])

In [ ]:
trainer.save_model('baseline_Roberta')

In [ ]:
with open('drive/MyDrive/ml_homework/hw4/data//train-test.txt', 'r', encoding='utf-8') as f:
    text_data = f.readlines()

In [ ]:
!pip install nltk

In [ ]:
import nltk
nltk.download('punkt_tab')

sentences = nltk.sent_tokenize(text_data[0], language='russian')

Output hidden; open in https://colab.research.google.com to view.

In [42]:
len(sentences)

29056

In [43]:
import re

def preprocess_sentences(sentences):
    sentences_no_punct = []
    for sentence in sentences:
        sentence = re.sub(r'[^\w\s]', '', sentence)
        sentences_no_punct.append(sentence)

    sentences_filtered = []
    for sentence in sentences_no_punct:
      if len(sentence.split()) > 2:
        sentences_filtered.append(sentence)
    return sentences_filtered

sentences = preprocess_sentences(sentences)
print(sentences)
len(sentences)

Output hidden; open in https://colab.research.google.com to view.

In [44]:
text_X_train, text_X_test = train_test_split(sentences, test_size=0.3)

In [56]:
text_X_train_dataset = Dataset.from_dict({'data': text_X_train})
text_X_test_dataset = Dataset.from_dict({'data': text_X_test})

In [49]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert/distilroberta-base")

In [55]:
def preprocess_function(examples):
    return tokenizer([" ".join(x) for x in examples["data"]])

In [54]:
text_X_train_dataset

Dataset({
    features: ['train_data'],
    num_rows: 18321
})

In [58]:
tokenized_train = text_X_train_dataset.map(
    preprocess_function,
    batched=True,
    num_proc=4,
    remove_columns=["data"],
)

Map (num_proc=4):   0%|          | 0/18321 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (594 > 512). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (867 > 512). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (627 > 512). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (540 > 512). Running this sequence through the model will result in indexing errors


In [59]:
tokenized_test = text_X_test_dataset.map(
    preprocess_function,
    batched=True,
    num_proc=4,
    remove_columns=["data"],
)

Map (num_proc=4):   0%|          | 0/7853 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (618 > 512). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (1044 > 512). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (653 > 512). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (795 > 512). Running this sequence through the model will result in indexing errors


In [60]:
block_size = 128

def group_texts(examples):
    # Concatenate all texts.
    concatenated_examples = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = len(concatenated_examples[list(examples.keys())[0]])
    # We drop the small remainder, we could add padding if the model supported it instead of this drop, you can
    # customize this part to your needs.
    if total_length >= block_size:
        total_length = (total_length // block_size) * block_size
    # Split by chunks of block_size.
    result = {
        k: [t[i : i + block_size] for i in range(0, total_length, block_size)]
        for k, t in concatenated_examples.items()
    }
    return result

In [61]:
lm_dataset_train = tokenized_train.map(group_texts, batched=True, num_proc=4)
lm_dataset_test = tokenized_test.map(group_texts, batched=True, num_proc=4)

Map (num_proc=4):   0%|          | 0/18321 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/7853 [00:00<?, ? examples/s]

In [62]:
from transformers import DataCollatorForLanguageModeling

tokenizer.pad_token = tokenizer.eos_token
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm_probability=0.15)

In [63]:
from transformers import AutoModelForMaskedLM

model = AutoModelForMaskedLM.from_pretrained("distilbert/distilroberta-base")

Some weights of the model checkpoint at distilbert/distilroberta-base were not used when initializing RobertaForMaskedLM: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [67]:
training_args = TrainingArguments(
    output_dir="my_awesome_eli5_mlm_model",
    eval_strategy="epoch",
    learning_rate=2e-5,
    num_train_epochs=3,
    weight_decay=0.01,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    fp16=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=lm_dataset_train,
    eval_dataset=lm_dataset_test,
    data_collator=data_collator,
    tokenizer=tokenizer
)

trainer.train()

<ipython-input-67-25b416a32511>:12: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss
1,0.686000,0.565938
2,0.597500,0.523810
3,0.572300,0.508304


TrainOutput(global_step=2475, training_loss=0.6107522367226957, metrics={'train_runtime': 720.5372, 'train_samples_per_second': 109.856, 'train_steps_per_second': 3.435, 'total_flos': 2624419775197440.0, 'train_loss': 0.6107522367226957, 'epoch': 3.0})

In [68]:
trainer.save_model('mln')

In [69]:
import math

eval_results = trainer.evaluate()
print(f"Perplexity: {math.exp(eval_results['eval_loss']):.2f}")

Perplexity: 1.66


In [70]:
model2 = AutoModelForSequenceClassification.from_pretrained("./mln", num_labels=2)

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at ./mln and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [71]:
training_args = TrainingArguments(
    output_dir="/test_trainer",
    eval_strategy="epoch",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    fp16=True,
    dataloader_num_workers=6,
    gradient_accumulation_steps=2
)

trainer = Trainer(
    model=model2,
    args=training_args,
    train_dataset=X_train,
    eval_dataset=X_test
)

In [72]:
trainer.train()

/usr/local/lib/python3.10/dist-packages/torch/utils/data/dataloader.py:617: UserWarning: This DataLoader will create 6 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


Epoch,Training Loss,Validation Loss
0,No log,0.218248
2,No log,0.196588


/usr/local/lib/python3.10/dist-packages/torch/utils/data/dataloader.py:617: UserWarning: This DataLoader will create 6 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


TrainOutput(global_step=387, training_loss=0.22567413882075663, metrics={'train_runtime': 187.6879, 'train_samples_per_second': 66.11, 'train_steps_per_second': 2.062, 'total_flos': 1638356786577408.0, 'train_loss': 0.22567413882075663, 'epoch': 2.988416988416988})

In [73]:
predictions = trainer.predict(X_test)

In [74]:
predicted_labels = predictions.predictions.argmax(-1)

In [75]:
predicted_labels

array([0, 1, 0, ..., 1, 1, 1])

In [76]:
X_test_labels = X_test['label']

In [77]:
accuracy = accuracy_score(X_test_labels, predicted_labels)
f1 = f1_score(X_test_labels, predicted_labels)
conf_matrix = confusion_matrix(X_test_labels, predicted_labels)

In [78]:
accuracy

0.904117315284828

In [79]:
f1

0.9046015712682379

In [80]:
conf_matrix

array([[797,  68],
       [102, 806]])

In [81]:
trainer.save_model('final_model')